<img src='https://gitlab.eumetsat.int/eumetlab/oceans/ocean-training/tools/frameworks/-/raw/main/img/MSOC_banner.png' align='right' width='100%'/>

<a href="../Index.ipynb">Index</a>

**Copyright:** 2025 MSOC training team <br>
**License:** MIT <br>
**Authors:** Ben Loveday (EUMETSAT/Innoflair UG), Hayley Evers-King (EUMETSAT), Anna Windle (NASA), Carina Poulin (NASA)

<div class="alert alert-block alert-success">
<h3>Multi-sensor Ocean Colour Course</h3></div>

<div class="alert alert-block alert-warning">
    
<b>PREREQUISITES </b>
    
This notebook has the following prerequisites:
- **<a href="https://data.marine.copernicus.eu/register" target="_blank">A Copernicus Marine Service account</a>** to download data form the Copernicus Marine Service Data Store
- You should have built and activated the appropriate `msoc` Python environment in either your command line, or in the Anaconda navigator prior to launching this notebook.

Optional:

- **<a href="https://identity.dataspace.copernicus.eu/auth/realms/CDSE/login-actions/registration?client_id=account-console&tab_id=7mxjCv4mJxw" target="_blank">A Copernicus Data Space Ecosystem account</a>** if you want to download new Sentinel-2 MSI data from the CDSE.
- **<a href="https://user.eumetsat.int/register" target="_blank">A EUMETSAT User Portal account</a>** if you want to download new Sentinel-3 OLCI marine data from the EUMETSAT Data Store
- **<a href="https://my.wekeo.eu/user-registration" target="_blank">A WEkEO account</a>** if you want to download new Sentinel-2 MSI or Sentinel-3 OLCI marine data from WEkEO.
- **<a href="https://urs.earthdata.nasa.gov/" target="_blank">An Earthdata account</a>** if you want to download new PACE OCI data from Earthdata.

There are no prerequisite notebooks for this module.
</div>
<hr>

# Comparing biogeochemical and IOP products

| Dataset | Source / ID | product description |
|:--------------------:|:-----------------------:|:-------------:|
| North Atlantic Ocean Colour Plankton, Reflectance, Transparency and Optics, <br> L3 (daily) from Satellite Observations (Near Real Time) | CMEMS / OCEANCOLOUR_ATL_BGC_L3_NRT_009_111 | <a href="https://data.marine.copernicus.eu/product/OCEANCOLOUR_ATL_BGC_L3_NRT_009_111/description" target="_blank">Description</a> |
| Sentinel-3 OLCI level 2 full resolution (operational) | EUMETSAT Data Store / EO:EUM:DAT:0407 | <a href="https://user.eumetsat.int/catalogue/EO:EUM:DAT:SENTINEL-3:OL_2_WFR___NTC" target="_blank">Description</a> | EO:EUM:DAT:SENTINEL-3:OL_2_WFR___ | <a href="https://www.wekeo.eu/data?view=dataset&dataset=EO%3AEUM%3ADAT%3ASENTINEL-3%3AOL_2_WFR___" target="_blank">Description</a> |
| Sentinel-2 Level 1C Top of Atmosphere | CDSE / sentinel-2-l1c | <a href="https://documentation.dataspace.copernicus.eu/Data/SentinelMissions/Sentinel2.html#sentinel-2-level-1c-top-of-atmosphere-toa" target="_blank">Description</a> |
| PACE OCI Level-2 Regional Ocean Biogeochemical Properties Data | Earthdata / PACE_OCI_L2_BGC | <a href="https://cmr.earthdata.nasa.gov/search/concepts/C3620139680-OB_CLOUD.html" target="_blank">Description</a> |
| PACE OCI Level-2 Regional Inherent Optical Properties Data | Earthdata / PACE_OCI_L2_IOP | <a href="https://cmr.earthdata.nasa.gov/search/concepts/C3620139828-OB_CLOUD.html" target="_blank">Description</a> |

### Learning outcomes

At the end of this notebook you will;
* Be able load and visualise IOP and biogeochemical products from a variety of algorithms across our sensors of interest
* Be aware of the recommended flags for such products and how to apply them
* Know how to compare the different IOP and biogeochemical products that come from the various sensors and algorithm approaches, and interpret them
  

### Outline

The ultimate aim of ocean colour remote sensing is to use the information embedded in the signal to retrieve quantities of relevant properties of ocean waters. This can include inherent optical properties such as the absorption/scattering associated with various constituents (e.g. phytoplankton), or biogeochemical quantities such as chlorophyll-a concentration. In this notebook we'll look at the output products from a variety of algorithms designed to extract this information from Copernicus Sentinel-2 and 3 data, and NASA PACE data.

This notebook example uses data from the North Sea. A dynamic region influenced by sediments as a result of river discharge and tidal resuspension, as well as by seasonal and episodic phytoplankton blooms.

<div class="alert alert-info" role="alert">

## Importing dependencies

</div>

We begin by importing all of the libraries that we need to run this notebook. If you have built your python using the environment file provided in this repository, then you should have everything you need. For more information on building environment, please see the repository **<a href="../README.md" target="_blank">README</a>**.

In [ ]:
import cartopy                  # a library that supports mapping and projection
import cartopy.crs as ccrs      # a library that support mapping
import copernicusmarine         # a library to help us access CMEMS data
import os                       # a library that allows us access to basic operating system commands like making directories
from pathlib import Path        # a library that helps construct system path objects
import datetime                 # a library that allows us to work with dates and times
import xarray as xr             # a library that supports the use of multi-dimensional arrays in Python
import numpy as np              # a library that provides support for array-based mathematics
import matplotlib               # a library that support plotting
import matplotlib.pyplot as plt # a library that supports plotting
import dask                     # a library for parallel computing on large datasets efficiently
import glob                     # a library that helps us find files
import re                       # a library that supports pattern matching
import pandas as pd             # a library that helps us create DataFrames
import logging                  # a library that supports log-level management
import warnings                 # a library that helps us manage warnings

We will also set a few parameters to making running this notebook more convenient.

In [ ]:
# suppress warnings
warnings.filterwarnings('ignore')

# turn down copernicus marine verbosity
logging.getLogger("copernicusmarine").setLevel("ERROR")

# set default plot size
matplotlib.rcParams.update({'font.size': 12})

# memory management
dask.config.set({"array.slicing.split_large_chunks": True})
chunks={"rows": 512, "columns": 512}

Let's also get the Python ternary library (which was not included in the original environment file, but now is!).

In [ ]:
try:
    import ternary
except:
    !pip install python-ternary
    import ternary

Lastly, we will import a quick function we wrote ourselves that finds the largest common regular that fits within a supplied list of granules...it is a little long to include explicitly in the notebook.

In [ ]:
import common_box               # a bespoke function that finds a common regular box between many products
import ternary_diagram          # a bespoke function that exploits `python-ternary` to make ternary diagram plots

Everything is now imported, we can proceed...

<div class="alert alert-warning" role="alert">

## Defining functions

</div>

Next we will define some quick functions for re-use below. These will be used to respectively;

* extract dates from file names,
* flag MSI, OLCI and PACE OCI products,
* and embellish any output plots.

These cells are hidden by default but you can click on the arrow next to this cell to expand them. They will run automatically if you "restart and run all cells", but you will need to make sure you run them if you are clicking through cell-by-cell.

In [ ]:
def extract_datetime(filename):
    """
    Return YYYYMMDDTHHMMSS substring from a filename, or None if not found.

    Parameters
    ----------
    filename : str

    Returns
    -------
    str or None
    """
    match = re.search(r"\d{8}T\d{6}", filename)
    if match:
        return match.group(0)
    return None

In [ ]:
def flag_data(flags, flag_names, flag_values, my_flag_values, flag_bytes=True, dtype=None):
    """
    Return a mask indicating where selected flags are set.

    Parameters
    ----------
    flags : array-like
        Array of per-pixel flag values.
    flag_names : list of str
        Names of all available flags, aligned with `flag_values`.
    flag_values : array-like
        Numeric values for each flag (bitmasks or integer codes).
    my_flag_values : list of str
        Names of the flags to check for.
    flag_bytes : bool, optional
        If True, treat flags as bitwise masks; if False, match values directly.
    dtype : data-type, optional
        Optional dtype for the internal mask array.

    Returns
    -------
    ndarray
        Boolean mask if `flag_bytes=True`, otherwise an array of 0/1 values.
    """
    if not dtype:
        flag_bits = np.zeros(np.shape(flags), np.dtype(type(flags[0][0])))
    else:
        flag_bits = np.zeros(np.shape(flags), dtype)

    for flag in my_flag_values:
        if flag_bytes:
            try:
                flag_bits = flag_bits | flag_values[flag_names.index(flag)]
            except:
                print(flag + " not present")
        else:
            flag_bits[flags==flag_values[flag_names.index(flag)]] = 1

    if flag_bytes:
        return (flags & flag_bits) > 0
    else:
        return flag_bits

In [ ]:
def embellish_plot(m):
    """
    Add land features and labeled gridlines to a cartopy axis.

    Parameters
    ----------
    m : cartopy.mpl.geoaxes.GeoAxes
        Axis to embellish.

    Returns
    -------
    None
    """
    # Embellish with gridlines
    m.add_feature(cartopy.feature.NaturalEarthFeature('physical', 'land', '10m', edgecolor='k', facecolor='#7E9CA0', linewidth=0.5), zorder=500)
    g1 = m.gridlines(draw_labels = True, linestyle='--', linewidth=0.5, zorder=1000)
    g1.top_labels = g1.right_labels = False
    g1.xlabel_style = g1.ylabel_style = {'color': '0.5'}

<div class="alert alert-info" role="alert">

## Setting up the experiment

</div>

Next we will point to the data directory where the products we will use in this notebook are either stored, or will be stored. This notebook will check every matching product in his directory.

In [ ]:
data_dir = os.path.join(os.path.dirname(os.getcwd()), "Data", "Preprepared", "Day3", "BGC_IOP_comparison")
print(f"Data directory is: {data_dir}")
if not os.path.exists(data_dir):
    print("Please check you have put the example data in the correct place, or adapt the path above to your ROI or data")
else:
    print("Found data directory!")

Now let's set up some plotting options. There are three products for each sensor (OCI/OLCI/MSI) in the default data directory (`data_dir`), each one corresponding to a date. These will be loaded in date order. You can use the `select_slice` option to choose which product you want to look at in more detail (e.g. slice 0 is the first date). The `sensor_subsample` option sets the grid subsampling parameters for displaying spatial plots. This degrades the grid for OLCI and MSI, solely to control the memory we use when plotting. It does not affect any non-spatial plots.

Note that there are two products for OCI (BGC and IOP products - corresponding to the biogeochemical and IOP files) and three for OLCI (BAC, AAC, IOP - corresponding to the standard atmospheric correction, C2RCC atmospheric correction and IOP products). In the PACE OCI case, we deal with these separately as they are in separate files. In the OLCI case, we deal with these separately as we flag them differently.

Finally, we will set a few parameters that control general plotting, such as tick labels for log plots.

In [ ]:
select_slice = 0
sensor_subsample = {"OCI_BGC":1,
                    "OCI_IOP":1,
                    "OLCI_BAC":1,
                    "OLCI_AAC":1,                    
                    "OLCI_IOP":3,
                    "MSI":5}

log_ticklabels = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10]
log_ticks = np.log10(log_ticklabels)

Now we are all set, and can read in our data...

<div class="alert alert-info" role="alert">

## Preparing to load our level-2 ocean colour BGC and IOP products

</div>

Our products all have very different formats, so we will define a dictionary for each type that helps us read in the necessary data and decide what flags to use. As mentioned above, you can see that we have different options for reading in PACE OCI BGC/IOP products, mostly due to the fact that the data is distributed in separate files. For OLCI, we have three different banks of options;
* one for the output of the standard atmopsheric correction (BAC),
* one for the output of the C2RCC alternate atmopsheric correction for complex waters (AAC),
* and one for the IOP products.

For MSI, we have a single read as we have processed the data with `acolite`, which puts all of our level-2 products in the same L2W file (*Note that we have not included the L1R or L2R products in our data directory. You can find our processing configuration <a href="../ac_methods/acolite_configs/acolite_config.ini" target="_blank">here</a>*). Also you should note that we have chosen to process MSI with only a subset of the parameters that acolite allows, for a full list you should see the <a href="https://github.com/acolite/acolite/releases/tag/20250114.0" target="_blank">acolite manual</a>.

You don't need to worry too much about altering these cells unless you wish to alter the products you want to include, change the flags applied, or run with different atmopsheric correction approaches (in which case, you should review the approaches taken in the <a href="../Day2/AC_comparison.ipynb" target="_blank">atmospheric correction</a> notebook from day 2 of the course).

In [ ]:
# PACE OCI L2 biogeochemical products
OCI_BGC_opts = {"sensor": "OCI_BGC", "search_term": "*PACE*BGC*", "subfiles" : False, "lon": "longitude", "lat": "latitude", "vars": ["chlor_a"],
               "flag_var": "l2_flags", "flags": ["ATMFAIL", "LAND", "PRODWARN", "HIGLINT", "HILT", "HISATZEN", "CLDICE", "HISOLZEN", "CHLFAIL", "LAND"]}

In [ ]:
# PACE OCI L2 inherent optical property products
OCI_IOP_opts = {"sensor": "OCI_IOP", "search_term": "*PACE*IOP*", "subfiles" : False, "lon": "longitude", "lat": "latitude", "vars": ["bbp_442", "aph", "adg_442"],
                "flag_var": "l2_flags", "flags": ["ATMFAIL", "LAND", "PRODWARN", "HIGLINT", "HILT", "HISATZEN", "CLDICE", "CHLFAIL", "LAND"],
                "aph_channel": 3}

In [ ]:
# OLCI products with flagging for CHL from the BAC atmopsheric correction
OLCI_BAC_opts = {"sensor": "OLCI_BAC", "search_term": "*.SEN3", "subfiles": True, "lon": "longitude", "lat": "latitude", "vars": ["CHL_OC4ME"],
                 "flag_var": "WQSF", "flags": ["CLOUD", "CLOUD_AMBIGUOUS", "CLOUD_MARGIN", "INVALID", "COSMETIC", "SATURATED", "SUSPECT", "HISOLZEN",
                                               "HIGHGLINT", "SNOW_ICE", "LAND", "AC_FAIL", "WHITECAPS", "ADJAC", "RWNEG_O2", "RWNEG_O3", "RWNEG_O4",
                                               "RWNEG_O5", "RWNEG_O6", "RWNEG_O7", "RWNEG_O8", "OC4ME_FAIL"]}

In [ ]:
# OLCI products with flagging for CHL from the AAC atmopsheric correction
OLCI_AAC_opts = {"sensor": "OLCI_AAC", "search_term": "*.SEN3", "subfiles": True, "lon": "longitude", "lat": "latitude", "vars": ["CHL_NN", "TSM_NN"],
                 "flag_var": "WQSF", "flags": ["CLOUD", "CLOUD_AMBIGUOUS", "CLOUD_MARGIN", "INVALID", "COSMETIC", "SATURATED", "SUSPECT","HISOLZEN",
                                               "HIGHGLINT", "SNOW_ICE", "LAND", "OCNN_FAIL"]}

In [ ]:
# OLCI products with flagging for IOPS from the BAC atmopsheric correction
OLCI_IOP_opts = {"sensor": "OLCI_IOP", "search_term": "*.SEN3", "subfiles": True, "lon": "longitude", "lat": "latitude",
                 "vars": ["OWC", "acdm_443", "acdom_443", "aphy_443", "bbp_443", "anw_443"],
                 "flag_var": "WQSF", "flags": ["CLOUD", "CLOUD_AMBIGUOUS", "CLOUD_MARGIN", "INVALID", "COSMETIC", "SATURATED", "SUSPECT",
                                               "HIGHGLINT", "SNOW_ICE", "LAND", "AC_FAIL", "WHITECAPS", "ADJAC", "RWNEG_O2", "RWNEG_O3", "RWNEG_O4",
                                               "RWNEG_O5", "RWNEG_O6", "RWNEG_O7", "RWNEG_O8", "IOP_LSD_FAIL"]}

In [ ]:
# MSI products with flagging derived from acolite
MSI_opts = {"sensor": "MSI", "search_term": "*_acolite", "subfiles": True, "lon": "lon", "lat": "lat", "vars": ["chl_oc2", "chl_oc3", "SPM_Nechad2016"],
            "flag_var": "l2_flags", "flags": ["flag_exponent_swir", "flag_exponent_cirrus", "flag_exponent_toa", "flag_exponent_negative",
                                              "flag_exponent_outofscene", "flag_exponent_mixed"]}

<div class="alert alert-danger" role="alert">

**WAIT! What do you notice about the OLCI file option?**

* Are the flags consistent between the BAC/AAC/IOP cases? If not, why not?
</div>

<div class="alert alert-info" role="alert">

## Loading our level-2 ocean colour BGC and IOP products

</div>

Ok! Now we are ready to read in our data. The cell below will do so, cycling through all the products and storing the output in a dictionary of xarrays, `df`. We will also extract the dates of acquisition from each product and store those in a separate, but similarly structured dictionary `acq_dates`

In [ ]:
df = {}
acq_dates = {}
for sensor_opts in [OCI_BGC_opts, OCI_IOP_opts, OLCI_BAC_opts, OLCI_AAC_opts, OLCI_IOP_opts, MSI_opts]:
    acq_dates[sensor_opts["sensor"]] = []
    df[sensor_opts["sensor"]] = []
    fnames = glob.glob(os.path.join(data_dir, sensor_opts["search_term"]))

    # sort fnames by date
    extracts = []
    for fname in fnames:
        extracts.append(extract_datetime(fname))
    fnames_sorted = [x for _, x in sorted(zip(extracts, fnames))]

    for fname in fnames_sorted:
        acq_date = datetime.datetime.strptime(extract_datetime(fname), "%Y%m%dT%H%M%S")
        acq_dates[sensor_opts["sensor"]].append(acq_date)
        print(f"Found: {os.path.basename(fname)} ({acq_date})")
        if sensor_opts["subfiles"]:
            df[sensor_opts["sensor"]].append(xr.open_mfdataset(glob.glob(os.path.join(fname, "*.nc")), chunks=chunks))
        else:
            target_file = glob.glob(os.path.join(data_dir, fname))[0]
            df[sensor_opts["sensor"]].append(xr.merge([xr.open_dataset(target_file, group=g, chunks=chunks) for g in ["navigation_data", "geophysical_data", "sensor_band_parameters"]]))

If everything went well, you should see an output list of 18 products; 6 for OCI, 9 for OLCI and 3 for MSI. If not, check to make sure that your `data_dir` points to the right place and contains the expected example data.

<div class="alert alert-info" role="alert">

## Check for overlapping box

</div>

We have 18 products from 3 different sensors, all of which have different grids and spatial resolutions. Of these, only MSI comes on a regularly projected tile grid (in this case UTM). To compare them, we are going to find the largest regularly shaped box (e.g. one that can be defined by lonmin/lonmax/latmin/latmax) that will fit in all 18 products. We have written a quick function, `largest_common_box_and_masks`, that will do this. We imported this function above as part of the `common_box` module.

Lets fun the analysis and see what box size we get. The function will also give us "masks" for each box, which allows us to perform arithmetic operations, such as averages, only on the points that are in scope.

In [ ]:
all_grids = []
for sensor_opts in [OCI_BGC_opts, OCI_IOP_opts, OLCI_BAC_opts, OLCI_AAC_opts, OLCI_IOP_opts, MSI_opts]:
    for item in df[sensor_opts["sensor"]]:
        all_grids.append((item[sensor_opts["lat"]].values, item[sensor_opts["lon"]].values))

bbox, masks = common_box.largest_common_box_and_masks(all_grids)

print(f"Largest common rectangle (lat/lon): {[str(i) for i in bbox]}")

Now we have a common box, we will store the masks in a dictionary that matches the structure of our product dictionary `df`. This makes them easy to use later on.

In [ ]:
mask_count = 0
box_masks = {}
for sensor_opts in [OCI_BGC_opts, OCI_IOP_opts, OLCI_BAC_opts, OLCI_AAC_opts, OLCI_IOP_opts, MSI_opts]:
    box_masks[sensor_opts["sensor"]] = []
    for item in df[sensor_opts["sensor"]]:
        box_masks[sensor_opts["sensor"]].append(masks[mask_count])
        mask_count = mask_count + 1

Lets check our "co-coverage" box to see if it looks sensible...

In [ ]:
# subsample this plot to save memory
box_grid_pad = 15
box_subsample = 10

#plot
fig, ax = plt.subplots(1, 1, figsize=(8, 8), dpi=150, subplot_kw={"projection": ccrs.PlateCarree()})
for sensor_opts, cmap, zorder in zip([OCI_BGC_opts, OLCI_BAC_opts, MSI_opts], ["viridis", "cividis", "magma"], [0,1,2]):
    for item in df[sensor_opts["sensor"]]:
        ax.pcolormesh(item[sensor_opts["lon"]][::box_subsample,::box_subsample],
                       item[sensor_opts["lat"]][::box_subsample,::box_subsample],
                       item[sensor_opts["lon"]][::box_subsample,::box_subsample], cmap=cmap, zorder=zorder)
ax.set_xlim(bbox[2] - box_grid_pad, bbox[3] + box_grid_pad)
ax.set_ylim(bbox[0] - box_grid_pad, bbox[1] + box_grid_pad)
ax.plot([bbox[2], bbox[3], bbox[3], bbox[2], bbox[2]], [bbox[0], bbox[0], bbox[1], bbox[1], bbox[0]], 'r--')
ax.set_title("Granule coverage and maximum co-bounding box [MSI (magma], OLCI (cividis), OCI (viridis)", fontsize=12)
embellish_plot(ax);

You should see a red-dashed line around the common box, which more than likely closely matches the edges of the MSI granules (the small purple/yellow squares). If you want to look in closer, you can drop the `box_grid_pad` to do so. This plot also clearly shows the difference in swath size between the three instruments, with PACE OCI (green to yellow) having the widest viewing angle.

<div class="alert alert-info" role="alert">

## Quality flagging our data

</div>

We have read in our data and defined our common box and its masks, but we have not yet created any flag masks. In a similar approach to how we read in the data, lets read through each product, ingest the flags and create a mask based on them. We will, again, save this in a dictionary called `flag_masks`.

In [ ]:
flag_masks = {}
for sensor_opts in [OCI_BGC_opts, OCI_IOP_opts, OLCI_BAC_opts, OLCI_AAC_opts, OLCI_IOP_opts, MSI_opts]:
    print(f"Calculating: {sensor_opts['sensor']} flag mask")
    flag_bytes=True
    flag_masks[sensor_opts["sensor"]] = []
    for item in df[sensor_opts["sensor"]]:
        if "MSI" in sensor_opts["sensor"]:
            flag_bytes=False
            flag_names = []
            flag_values = []
            for i in item.attrs:
                if "flag" in i and not "viirs" in i:
                    flag_names.append(i)
                    flag_values.append(item.attrs[i])
        else:
            flag_names = item[sensor_opts["flag_var"]].attrs["flag_meanings"].split(' ')
            flag_values = np.array(item[sensor_opts["flag_var"]].attrs["flag_masks"], dtype=np.uint64)
        flag_mask = flag_data(np.array(item[sensor_opts["flag_var"]], dtype=np.uint64),
                           flag_names,
                           flag_values,
                           sensor_opts["flags"],
                           flag_bytes=flag_bytes).astype(float)
        flag_mask[flag_mask == 1.0] = np.nan
        flag_mask[flag_mask == 0.0] = 1.0
        flag_masks[sensor_opts["sensor"]].append(flag_mask)

Now that we have all of our masks (both box and quality) defined, we can apply them to our variables of interest, removing all data we don't want. Amongst other things, this enables us to take mean values of a common area that we can compare with other data sources, such as those from long time series gridded level-3 BGC and IOP products available from the Copernicus Marine Service.

So lets do this, applying the appropriate `flag_mask` and `box_mask` to each **BGC** variable in turn and taking the mean (ignoring area weighting in this case). This will give us a single point value for each product for each acquisition date.

In [ ]:
variables = {}
for sensor_opts in [OCI_BGC_opts, OLCI_BAC_opts, OLCI_AAC_opts, MSI_opts]:
    variables[sensor_opts["sensor"]] = {}
    for var in sensor_opts["vars"]:
        print(f"Sensor: {sensor_opts['sensor']}, var: {var}")
        variables[sensor_opts["sensor"]][var] = []
        for item, flag_mask, box_mask in zip(df[sensor_opts["sensor"]], flag_masks[sensor_opts["sensor"]], box_masks[sensor_opts["sensor"]]):
            if "Nechad" in var:
                matches = [s for s in item.variables if "Nechad" in s]
                read_var = matches[0]
            else:
                read_var = var
            variables[sensor_opts["sensor"]][var].append(np.nanmean(flag_mask*box_mask*item[read_var]))

Our level-2 biogeochemical products are now prepared. Lets move on to our level-3 products.

(*Note: if you didn't spot it above, we are using two algorithms, chl_oc2 and chl_oc3, to derive chlorophyll concentration from MSI, and the SPM_Nechad2016 algorithm for SPM. Feel free to preprocess any data with different products for comparison. You can find a full list of L2W products <a href="https://github.com/acolite/acolite/releases/tag/20250114.0" target="_blank">here</a>.*)

<div class="alert alert-info" role="alert">

## Fetching level-3 ocean colour products from the Copernicus Marine Service

</div>

We will get our CMEMS products using exactly the same approach as we did on Day 1 of the course. See the <a href="../Day1/Access_multisensor_CMEMS_Data_Store.ipynb" target="_blank">Accessing CMEMS data notebook</a> if you need a reminder!

We start by creating a credentials file to support authorisation of the service.

In [ ]:
# Default location expected by the copernicusmarine package
copernicus_marine_credentials_file = Path(Path.home() / '.copernicusmarine' / '.copernicusmarine-credentials')

# Create it only if it does not already exists
if not copernicus_marine_credentials_file.is_file():
    copernicusmarine.login()

Now we will define the products that parameters of the products that we wish to download. In this case, these are the 1 km, multi-sensor plankton and transparency products, so we can look at the chlorophyll and suspended particular matter concentrations.

In [ ]:
products = []
products.append({"region" : "atl", "product" : "plankton", "rec": "my", "sensors" : "multi", "resolution" : "1km", "timing" : "1D"})
products.append({"region" : "atl", "product" : "transp", "rec": "my", "sensors" : "multi", "resolution" : "1km", "timing" : "1D"})

We will need to specify the variables names for chlorophyll ['CHL'] and suspended particulate matter ['SPM'].

In [ ]:
CMEMS_variables = ['CHL', 'SPM']

Lastly, we need to subset our search in time and space. Our spatial subset is set by the "common box" we defined earlier, but we have not yet set our time bounds. Lets do this now;

*Note: we are using `datetime.datetime.now()` to get the most up to date data, but we are using the `my` product type, so it may not be up to today.*

In [ ]:
# time filter for matching products
dtstart = datetime.datetime(2024, 1, 1, 0, 0)
dtend = datetime.datetime.now()

We are ready; lets get the final, level-3, products we need for this notebook...

In [ ]:
output_filenames = {}

for product, CMEMS_variable in zip(products, CMEMS_variables):
    CMEMS_product = f"cmems_obs-oc_{product['region']}_bgc-{product['product']}_{product['rec']}_l3-{product['sensors']}-{product['resolution']}_P{product['timing']}"
    output_filenames[CMEMS_variable] = os.path.join(os.getcwd(), data_dir, f"{CMEMS_product}_{dtstart.strftime('%Y%m%d')}_{dtend.strftime('%Y%m%d')}.nc")
        
    if os.path.exists(output_filenames[CMEMS_variable]):
        print(f"Skipping: {CMEMS_product} as it exists")
    else:
        print(f"Fetching: {CMEMS_product}")
        copernicusmarine.subset(
        dataset_id=CMEMS_product,
        variables=[CMEMS_variable],
        minimum_longitude=bbox[2],
        maximum_longitude=bbox[3],
        minimum_latitude=bbox[0],
        maximum_latitude=bbox[1],
        start_datetime=dtstart.strftime("%Y-%m-%dT00:00:00.000Z"),
        end_datetime=dtend.strftime("%Y-%m-%dT00:00:00.000Z"),
        output_filename=output_filenames[CMEMS_variable]);

<div class="alert alert-info" role="alert">

## Comparing biogeochemical variables

</div>

We're finally ready to do some comparison! Let's start by looking at a few of the available biogeochemical parameters, starting with the easiest to work with - those from CMEMS! First step, read in the data....

In [ ]:
df_CMEMS = {}
for CMEMS_variable in CMEMS_variables:
    df_CMEMS[CMEMS_variable] = xr.open_dataset(output_filenames[CMEMS_variable], chunks=chunks)

Now lets plot the mean and standard deviation of our variables for our time window....

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(20, 5), dpi=150, subplot_kw={"projection": ccrs.PlateCarree()})

for ax, CMEMS_variable, cmap, label in zip(axs, CMEMS_variables, ["viridis", "plasma"], ["CHL mg.m$^{-3}$", "SPM mg.m$^{-3}$"]):
    plt.sca(ax)
    plt.pcolormesh(df_CMEMS[CMEMS_variable].longitude, df_CMEMS[CMEMS_variable].latitude,
                   np.log10(df_CMEMS[CMEMS_variable].mean(dim="time")[CMEMS_variable]),
                   cmap=cmap, vmin=np.log10(1), vmax=np.log10(5))
    cbar = plt.colorbar(label=label, ticks=log_ticks)
    cbar.set_ticklabels(log_ticklabels)
    plt.title(f"Mean {CMEMS_variable}: {dtstart.strftime('%Y%m%d')} to {dtend.strftime('%Y%m%d')}")
    embellish_plot(ax)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(20, 5), dpi=150, subplot_kw={"projection": ccrs.PlateCarree()})

for ax, CMEMS_variable, cmap, label in zip(axs, CMEMS_variables, ["viridis", "plasma"], ["CHL mg.m$^{-3}$", "SPM mg.m$^{-3}$"]):
    plt.sca(ax)
    plt.pcolormesh(df_CMEMS[CMEMS_variable].longitude, df_CMEMS[CMEMS_variable].latitude,
                   np.log10(df_CMEMS[CMEMS_variable].std(dim="time")[CMEMS_variable]),
                   cmap=cmap, vmin=np.log10(1), vmax=np.log10(5))
    cbar = plt.colorbar(label=label, ticks=log_ticks)
    cbar.set_ticklabels(log_ticklabels)
    plt.title(f"Std. dev. {CMEMS_variable}: {dtstart.strftime('%Y%m%d')} to {dtend.strftime('%Y%m%d')}")
    embellish_plot(ax)

This simple visualisation shows the mean and standard deviation of chlorophyll-a and SPM concentrations derived from the CMEMS Atlantic region multi-year 1 km data for our common box. The chlorophyll-a is a blended algorithm of OC5 (Gohin et al. 2002) and CI (Hu et al. 2012) algorithms (Garnesson et al., 2019), whilst the SPM is derived from Gohin et al, 2011. OC5 is designed to support the derivation of Chlorophyll-a in complex coastal waters, whilst the blending with CI ensures low chlorophyll-a waters can also be captured. The SPM is derived from the method of Gohin et al., 2011 using Rrs at 550 nm and a relationship with chlorophyll-a to extract the non-algal SPM component. Knowing the dynamics of this area, and these two methods - do you expect these two algorithms to perform well in this area? Do they show similar patterns? Would you expect this?

Next, lets take a look at the data for each of these variables from each of our sensors, starting with chlorophyll...

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(15, 12), dpi=150, subplot_kw={"projection": ccrs.PlateCarree()})
    
for sensor_opts, plot_var, ax in zip([OCI_BGC_opts, OLCI_BAC_opts, OLCI_AAC_opts, MSI_opts], ["chlor_a", "CHL_OC4ME", "CHL_NN", "chl_oc3"], axs.flatten()):

    ss = sensor_subsample[sensor_opts["sensor"]]
    if "CHL_" in plot_var:
        transform_var = df[sensor_opts["sensor"]][select_slice][plot_var][::ss, ::ss]
    else:
        transform_var = np.log10(df[sensor_opts["sensor"]][select_slice][plot_var][::ss, ::ss])
        
    p1 = ax.pcolormesh(df[sensor_opts["sensor"]][select_slice][sensor_opts["lon"]][::ss, ::ss],
                       df[sensor_opts["sensor"]][select_slice][sensor_opts["lat"]][::ss, ::ss],
                       transform_var *
                       flag_masks[sensor_opts["sensor"]][select_slice][::ss, ::ss] *
                       box_masks[sensor_opts["sensor"]][select_slice][::ss, ::ss],
                       cmap=plt.cm.viridis, vmin=np.log10(1), vmax=np.log10(10),
                       transform=ccrs.PlateCarree())

    ax.annotate(sensor_opts["sensor"].split('_')[0], (0.02, 0.02), xycoords="axes fraction", color="w", fontsize=20)
    ax.set_xlim([bbox[2], bbox[3]])
    ax.set_ylim([bbox[0], bbox[1]])
    embellish_plot(ax)
    
    cbar = plt.colorbar(p1, orientation="horizontal", label=f"{plot_var} [mg.m$^{-3}$] ({sensor_opts['sensor'].split('_')[0]})", ticks=log_ticks)
    cbar.set_ticklabels(log_ticklabels)

<div class="alert alert-danger" role="alert">

**WAIT! What do you notice about these plots?**

* Three of the four algorithms above take relatively similar approaches to deriving chlorophyll-a. Can you guess which?!

* The **chlor_a** algorithm for PACE and OC4ME for Sentinel-3 use similar combinations of bands from the blue and green parts of the spectrum, along with a CI switch to estimate chlorophyll-a for the latter. OC3 applied to the Sentinel-2 data also used similar bands. Yet, we see quite different results. Knowing what you know about these sensors, can you think why we might have these differences?

* Interestingly, the most different algorithm here (the CHL_NN) applied to OLCI is perhaps the most "median" result of the 4. Do you think this algorithm is performing well?

* Without validation or some further investigation, it's really hard to be specific about the performance of each algorithm. However given the optical complexity of this region, the appropriate training data set and the signal:noise and calibration status of Sentinel-3 OLCI, it could be expected that the CHL_NN performs well in such waters. Why could it be that we don't see the same feature in the OLCI-CHL_NN, that we see in the left of the images from PACE-OC4/OLCI-OC4ME/MSI-chl_oc3?

* The fact that the other algorithms are relatively similar in their form, but produce different results, could indicate that we need to carefully consider the sensors themselves more to understand the results. What are the differences between PACE, OLCI, and MSI that might explain this?

</div>

Let's see what other parameters we can look at the understand the underlying optics and sensor performance in these scenes...

We will begin by plotting our other biogeochemical variable, SPM.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(20, 10), dpi=150, subplot_kw={"projection": ccrs.PlateCarree()})
    
for sensor_opts, plot_var, ax in zip([OLCI_AAC_opts, MSI_opts], ["TSM_NN", "SPM_Nechad2016_665"], axs):
    
    ss = sensor_subsample[sensor_opts["sensor"]]
    if "TSM_NN" in plot_var:
        transform_var = df[sensor_opts["sensor"]][select_slice][plot_var][::ss, ::ss]
    else:
        try:
            transform_var = np.log10(df[sensor_opts["sensor"]][select_slice][plot_var][::ss, ::ss])
        except:
            transform_var = np.log10(df[sensor_opts["sensor"]][select_slice]["SPM_Nechad2016_667"][::ss, ::ss])
            
    p1 = ax.pcolormesh(df[sensor_opts["sensor"]][select_slice][sensor_opts["lon"]][::ss, ::ss],
                       df[sensor_opts["sensor"]][select_slice][sensor_opts["lat"]][::ss, ::ss],
                       transform_var *
                       flag_masks[sensor_opts["sensor"]][select_slice][::ss, ::ss] *
                       box_masks[sensor_opts["sensor"]][select_slice][::ss, ::ss],
                       cmap=plt.cm.inferno, vmin=np.log10(1), vmax=np.log10(10),
                       transform=ccrs.PlateCarree())

    ax.annotate(sensor_opts["sensor"].split('_')[0], (0.02, 0.02), xycoords="axes fraction", color="w", fontsize=20)
    ax.set_xlim([bbox[2], bbox[3]])
    ax.set_ylim([bbox[0], bbox[1]])
    embellish_plot(ax)

    cbar = plt.colorbar(p1, orientation="horizontal", label=f"{plot_var} [mg.m$^{-3}$] ({sensor_opts['sensor'].split('_')[0]})", ticks=log_ticks)
    cbar.set_ticklabels(log_ticklabels)

<div class="alert alert-danger" role="alert">

**WAIT! What do you notice about these plots?**

* The OLCI neural network (NN) processor co-solves for the TSM alongside chlorophyll-a. Perhaps the most interesting part of this image, is that we now see a feature on the left, which was indicated in the chlorophyll-a band ratio derived products in the previous example. Is the NN appropriately partitioning the chlorophyll and non-algal SPM in this case?

* Do you think it's better to have these parameters derived simultaneously or independently? Can you think of any advantages and disadvantages?

* What about the Sentinel-2 example with the algorithm of Nechad applied. Do you see similarities? Do you think Sentinel-2 would be able to do a better job estimating SPM than TSM in this area? Why?

(Bonus points for anyone who can remember why there's stripes!)

</div>

Looking at individual images can be a very useful exercise to think about combined sensor and algorithm performance. However individual images are subject to various uncertainties, and may not be representative of performance over time. By taking averages across our common box, we can look at multiple examples, in the context of a time series we derive from the CMEMS level-3 data. We can use this to learn more about regional variability and the performance of these different approaches.

Lets now plot our individual scenes as a single point, averaged across each box, against the CMEMS data....

*Note: you can use the `day_pad` argument to zoom in on the level-2 data..*

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(20, 5), dpi=150)

date_pad = None

for ax, CMEMS_variable, cmap, label in zip(axs, CMEMS_variables, ["g", "k"], ["CHL mg.m$^{-3}$", "SPM mg.m$^{-3}$"]):
    ax.plot(df_CMEMS[CMEMS_variable].time, np.log10(df_CMEMS[CMEMS_variable].mean(dim=["latitude", "longitude"])[CMEMS_variable]), c=cmap)
    ax.set_ylabel(label)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=270)
    ax.set_yticks(log_ticks, log_ticklabels)
    if date_pad:
        ax.set_xlim([min(acq_dates['MSI']) - datetime.timedelta(days=date_pad), max(acq_dates['MSI']) + datetime.timedelta(days=date_pad)])
    ax.set_title(f"Mean {CMEMS_variable}")
    
# add single sensor values

p1 = axs[0].scatter(acq_dates["MSI"], np.log10(variables["MSI"]["chl_oc2"]), marker="x", s=100, color='red')
p2 = axs[0].scatter(acq_dates["MSI"], np.log10(variables["MSI"]["chl_oc3"]), marker="x", s=100, color='blue')
p3 = axs[0].scatter(acq_dates["OLCI_BAC"], variables["OLCI_BAC"]["CHL_OC4ME"], marker="o", s=100, color='red')
p4 = axs[0].scatter(acq_dates["OLCI_AAC"], variables["OLCI_AAC"]["CHL_NN"], marker="o", s=100, color='blue')
p5 = axs[0].scatter(acq_dates["OCI_BGC"], np.log10(variables["OCI_BGC"]["chlor_a"]), marker="*", s=100, color='black')
leg = axs[0].legend([p1,p2,p3,p4,p5],["MSI: OC2", "MSI: OC3", "OLCI: CHL_OC4ME", "OLCI: CHL_NN", "OCI"], ncol=5, fontsize=10)
leg.set_frame_on(False)

p1 = axs[1].scatter(acq_dates["MSI"], np.log10(variables["MSI"]["SPM_Nechad2016"]), marker="x", s=100, color='red')
p2 = axs[1].scatter(acq_dates["OLCI_AAC"], variables["OLCI_AAC"]["TSM_NN"], marker="o", s=100, color='blue')
leg = axs[1].legend([p1,p2],["MSI: NECHAD", "OLCI: TSM_NN"], ncol=2, fontsize=10)
leg.set_frame_on(False);

<div class="alert alert-danger" role="alert">

**WAIT! What do you notice about these plots?**

* What do you observe about the performance of the different sensors and algorithms vs the time series from CMEMS data?
* Knowing the sensors and algorithms involved, is there an algorithm/sensor combination that you would expect to perform most consistently with the CMEMS time series?
</div>

<div class="alert alert-info" role="alert">

## Investigating inherent optical properties

</div>

Deriving IOPs from the reflectance, can help provide a more analytical approach to understanding and quantifying the optically significant constituents in the ocean waters of interest. Here we will look at a range of algorithms, largely of the semi-analytical type, that can be used to derive the absoorption and backscattering related properties of ocean waters. Our list of compared products is in the below;

| Source | Parameters | Reference |
|:--------------------:|:-----------------------:|:----------:|
| OLCI (standard; iop_lsd.nc) | acdm_443, acdom_443, anw_443, aphy_443, bbp_443 | Bonelli et al. (2021), Loisel et al. (2018), Zhang et al (modified) | 
| MSI (acolite) | qaa_v6_a(λ), qaa_v6_bbp(λ), | Lee et al. (2013)|
| OCI (standard; PACE IOP) | a(λ), adg_442, adg_s, a_w, aph(λ), bb(λ), bbp_s, bbw | Generalised IOP model (GIOP) from Werdell et al. (2013).|

Lets start our comparison with the only semi-consistent parameter we can get from all three sensors; bbp$_{443}$....

*Note: OLCI IOP products are provided at a single band, but the PACE OCI GIOP model gives us spectrally resolved values in some cases. We are extracting the GIOP wavelengths a the wavelength common to the OLCI parameter*

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(20, 10), dpi=150, subplot_kw={"projection": ccrs.PlateCarree()})

for sensor_opts, plot_var, ax, tag in zip([OCI_IOP_opts, OLCI_IOP_opts, MSI_opts], ["bbp_442", "bbp_443", "qaa_v6_bbp_443"], axs, ["GIOP", "Loisel", "QAA"]):

    ss = sensor_subsample[sensor_opts["sensor"]]
    if "OLCI" in sensor_opts["sensor"]:
        transform_var = df[sensor_opts["sensor"]][select_slice][plot_var][::ss, ::ss]
    else:
        transform_var = np.log10(df[sensor_opts["sensor"]][select_slice][plot_var][::ss, ::ss])
        
    p1 = ax.pcolormesh(df[sensor_opts["sensor"]][select_slice][sensor_opts["lon"]][::ss, ::ss],
                       df[sensor_opts["sensor"]][select_slice][sensor_opts["lat"]][::ss, ::ss],
                       transform_var *
                       flag_masks[sensor_opts["sensor"]][select_slice][::ss, ::ss] *
                       box_masks[sensor_opts["sensor"]][select_slice][::ss, ::ss],
                       cmap=plt.cm.viridis, vmin=np.log10(0.001), vmax=np.log10(0.1),
                       transform=ccrs.PlateCarree())

    ax.annotate(f"{sensor_opts['sensor'].split('_')[0]}: {tag}", (0.02, 0.02), xycoords="axes fraction", color="k", fontsize=20)
    ax.set_xlim([bbox[2], bbox[3]])
    ax.set_ylim([bbox[0], bbox[1]])
    embellish_plot(ax)

    cbar = plt.colorbar(p1, orientation="horizontal", label=f"{plot_var} [m$^{-1}$]", ticks=log_ticks)
    cbar.set_ticklabels(log_ticklabels)

<div class="alert alert-danger" role="alert">

**WAIT! What do you notice about these plots?**

* In the images above we see the particulate backscattering estimated at 442/443 nm. What similarities and differences do you see?
* What dynamics of the area do you see represented?
* What about the particular spectral location of this product might make it harder/easier to use certain sensors to derive it?
</div>

Let's look at a different IOP, aphy$_{442}$...

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(20, 10), dpi=150, subplot_kw={"projection": ccrs.PlateCarree()})
    
for sensor_opts, plot_var, ax, tag in zip([OCI_IOP_opts, OLCI_IOP_opts], ["aphy_442", "aphy_443"], axs, ["GIOP", "Zhang"]):

    ss = sensor_subsample[sensor_opts["sensor"]]
    if "OLCI" in sensor_opts["sensor"]:
        transform_var = df[sensor_opts["sensor"]][select_slice][plot_var][::ss, ::ss]
    elif "OCI_IOP" in sensor_opts["sensor"] and "aph" in plot_var: 
        transform_var = np.log10(df[sensor_opts["sensor"]][select_slice][plot_var][::ss, ::ss, sensor_opts["aph_channel"]])
    else:
        transform_var = np.log10(df[sensor_opts["sensor"]][select_slice][plot_var][::ss, ::ss])

    p1 = ax.pcolormesh(df[sensor_opts["sensor"]][select_slice][sensor_opts["lon"]][::ss, ::ss],
                       df[sensor_opts["sensor"]][select_slice][sensor_opts["lat"]][::ss, ::ss],
                       transform_var *
                       flag_masks[sensor_opts["sensor"]][select_slice][::ss, ::ss] *
                       box_masks[sensor_opts["sensor"]][select_slice][::ss, ::ss],
                       cmap=plt.cm.viridis, vmin=np.log10(0.01), vmax=np.log10(0.1),
                       transform=ccrs.PlateCarree())

    ax.annotate(f"{sensor_opts['sensor'].split('_')[0]}: {tag}", (0.02, 0.02), xycoords="axes fraction", color="k", fontsize=20)
    ax.set_xlim([bbox[2], bbox[3]])
    ax.set_ylim([bbox[0], bbox[1]])
    embellish_plot(ax)

    cbar = plt.colorbar(p1, orientation="horizontal", label=f"{plot_var} [m$^{-1}$]", ticks=log_ticks)
    cbar.set_ticklabels(log_ticklabels)

<div class="alert alert-danger" role="alert">

**WAIT! What do you notice about these plots?**

* Here we have two estimates of phytoplankton absorption at 443nm. Do you see any consistency? Any idea why there might be some differences? 
* Why do you think we don't have a Sentinel-2 product for this parameter?
</div>

Finally, let's look at the absorption related to "dissolved components", adg$_{442}$ and acdm$_{443}$. Note that these values are not quite equivalent, but here we will assume that they approximate each other.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(20, 10), dpi=150, subplot_kw={"projection": ccrs.PlateCarree()})
    
for sensor_opts, plot_var, ax, tag in zip([OCI_IOP_opts, OLCI_IOP_opts], ["adg_442", "acdm_443"], axs, ["GIOP", "Zhang"]):

    ss = sensor_subsample[sensor_opts["sensor"]]
    if "OLCI" in sensor_opts["sensor"]:
        transform_var = df[sensor_opts["sensor"]][select_slice][plot_var][::ss, ::ss]
    else:
        transform_var = np.log10(df[sensor_opts["sensor"]][select_slice][plot_var][::ss, ::ss])

    p1 = ax.pcolormesh(df[sensor_opts["sensor"]][select_slice][sensor_opts["lon"]][::ss, ::ss],
                       df[sensor_opts["sensor"]][select_slice][sensor_opts["lat"]][::ss, ::ss],
                       transform_var *
                       flag_masks[sensor_opts["sensor"]][select_slice][::ss, ::ss] *
                       box_masks[sensor_opts["sensor"]][select_slice][::ss, ::ss],
                       cmap=plt.cm.viridis, vmin=np.log10(0.01), vmax=np.log10(0.3),
                       transform=ccrs.PlateCarree())

    ax.annotate(f"{sensor_opts['sensor'].split('_')[0]}: {tag}", (0.02, 0.02), xycoords="axes fraction", color="k", fontsize=20)
    ax.set_xlim([bbox[2], bbox[3]])
    ax.set_ylim([bbox[0], bbox[1]])
    embellish_plot(ax)

    cbar = plt.colorbar(p1, orientation="horizontal", label=f"{plot_var} [m$^{-1}$]", ticks=log_ticks)
    cbar.set_ticklabels(log_ticklabels)

<div class="alert alert-danger" role="alert">

**WAIT! What do you notice about these plots?**

* Here we have two estimates of CDOM absorption at 442/443 nm. These images look very different...does this connect with what we've seen before?
* Do you think this is related to differences in the algorithms, differences in the sensors, or something else?
</div>

<div class="alert alert-info" role="alert">

## Making IOP ternary plots

</div>

It can be very useful to think about these different components in combination, and how they relate to each other. A classic way of doing this in situ is using a ternary diagram. Here we show this for the OLCI IOP product suite. Note that we are not using an a$_{nap}$ product from the OLCI product suite, as it does not provide one. We are calculating a$_{nap}$ using the following relationship taken from Forge et al., (2021);

a$_{nap}$(λ) = a$_{cdm}$(λ) - a$_{cdom}$(λ)

We are taking a look at the whole scene, not just the data in our box of interest. To reduce the memory load for this plot, we are subsampling the data by the `sensor_subsample` parameter we set at the top of this notebook. Alongside the IOP products, we will also read in the **optical water type classification** that can be found in the **iop_lsd.nc** file of the standard level-2 product.

Lets prepare our data...

In [ ]:
ss = sensor_subsample["OLCI_IOP"]

a_cdom = df["OLCI_IOP"][select_slice]["acdom_443"][::ss, ::ss] * flag_masks["OLCI_IOP"][select_slice][::ss, ::ss]
a_phy = df["OLCI_IOP"][select_slice]["aphy_443"][::ss, ::ss] * flag_masks["OLCI_IOP"][select_slice][::ss, ::ss]
a_cdm = df["OLCI_IOP"][select_slice]["acdm_443"][::ss, ::ss] * flag_masks["OLCI_IOP"][select_slice][::ss, ::ss]
OWC = df["OLCI_IOP"][select_slice]["OWC"][::ss, ::ss] * flag_masks["OLCI_IOP"][select_slice][::ss, ::ss]

OWC = np.array(OWC).ravel()
a_cdom = 10**np.array(a_cdom).ravel()
a_phy = 10**np.array(a_phy).ravel()
a_cdm = 10**np.array(a_cdm).ravel()
a_nap = a_cdm - a_cdom

# remove nans
a_cdom[a_cdom < 0.0] = np.nan
a_phy[a_phy < 0.0] = np.nan
a_nap[a_nap < 0.0] = np.nan

ii = np.where(np.isfinite(a_cdom) & np.isfinite(a_nap) & np.isfinite(a_phy))
a_phy = a_phy[ii]
a_nap = a_nap[ii]
a_cdom = a_cdom[ii]
OWC = OWC[ii]

# sort by OWC low to high to help plotting
sorted_indices = np.argsort(OWC)

a_phy_sorted = a_phy[sorted_indices]
a_nap_sorted = a_nap[sorted_indices]
a_cdom_sorted = a_cdom[sorted_indices]
OWC_sorted = OWC[sorted_indices]

Lets now make our ternary plot, colour coding each point on the diagram by the optical water type....

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(20, 10), dpi=150, subplot_kw={"projection": ccrs.PlateCarree()})

p1 = axs[0].pcolormesh(df["OLCI_IOP"][select_slice][sensor_opts["lon"]][::ss, ::ss],
                       df["OLCI_IOP"][select_slice][sensor_opts["lat"]][::ss, ::ss],
                       df["OLCI_IOP"][select_slice]["OWC"][::ss, ::ss] * 
                       flag_masks["OLCI_IOP"][select_slice][::ss, ::ss],
                       cmap=plt.cm.Spectral,
                       transform=ccrs.PlateCarree())
embellish_plot(axs[0])

cmap = plt.cm.viridis
norm = plt.Normalize(min(OWC), max(OWC))

subfig, tax = ternary_diagram.plot_ternary(
    a_phy_sorted, a_nap_sorted, a_cdom_sorted,
    labels=('a$_{phy}$', 'a$_{nap}$', 'a$_{cdom}$'),
    title=None,
    cmap="Spectral",
    colorby=OWC_sorted,
    size=1,
    alpha=0.5,
    axes_colors=('#37BC7D', '#FF8904', '#7B3306'),
    fontsize=15,
    ax=axs[1],
    clabel="OWC")

<div class="alert alert-danger" role="alert">

**WAIT! What do you notice about these plots?**

* What do the distribution of these water classes tell you about the water types present across the north sea?
* Looking at the ternary diagram, do you see relationships between the parameters?
* Where are they not so related? 
</div>

<div class="alert alert-success" role="alert">

## What next?

Adapt this script to read in data from your region of interest. What do you notice about how the different products perform? Do you have any feeling for why certain products may perform better or worth for each sensor given the optical characteristics of your particular region?

</div>

<hr>
<a href="../Index.ipynb">Index</a>
<hr>
<a href="https://gitlab.com/eo_training/msoc" target="_blank">View on GitLab</a> | <a href="https://training.eumetsat.int/" target="_blank">EUMETSAT Training</a> | <a href=mailto:ops@eumetsat.int target="_blank">Contact helpdesk for support </a> | <a href=mailto:training@eumetsat.int target="_blank">Contact our training team to collaborate on and reuse this material</a></span></p>